# Intermediate challenge: Person 1 (baseline + fewer qubits)

**Molecule:** LiH at 1.6 Å, `sto3g` basis. **Target:** stay within chemical accuracy (1.6 mHa) of the exact energy.

This notebook does two jobs:
1. **Baseline:** the tutorial's method applied unchanged to LiH, so we know what we're beating.
2. **Fewer qubits:** apply qubit-reduction tricks one at a time and record what each one buys.

All shared code is in `intermediate_challenge/p1_setup.py`. Teammates should import from it so everyone uses the same molecule, exact energy and resource counting:
```python
from intermediate_challenge.p1_setup import *
problem, mapper = reduced_lih("6q")   # or "4q" / "full"
```

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from intermediate_challenge.p1_setup import *

## 1. The problem and the exact answer
Every error we report is measured against this exact energy of the **full** molecule. That way, any accuracy lost by our reductions shows up honestly in the error column.

In [2]:
full = lih_problem()
E_EXACT = exact_energy(full)
print("Spatial orbitals :", full.num_spatial_orbitals)
print("Electrons (α, β) :", full.num_particles)
print("Orbital energies :", full.orbital_energies.round(3))
print(f"Exact energy     : {E_EXACT:.6f} Ha")

Spatial orbitals : 6
Electrons (α, β) : (2, 2)
Orbital energies : [-2.349 -0.285  0.078  0.164  0.164  0.548]
Exact energy     : -7.882324 Ha


## 2. Baseline: the tutorial method, unchanged
Jordan-Wigner + Hartree-Fock + UCCSD on the full problem. We record the resources but **don't run VQE** here: with 92 parameters on 12 qubits it takes far too long. That cost is exactly the problem this challenge is about.

In [3]:
results = []
p, m = reduced_lih("full")
op = m.map(p.hamiltonian.second_q_op())
row = measure_resources(build_ansatz(p, m), op, "Baseline (JW, 12q)")
row["reduced_exact_err"] = 0.0
row["vqe_err"] = None
results.append(row)
pd.DataFrame(results)

,config,qubits,params,CNOTs,depth,pauli_terms,meas_groups,reduced_exact_err,vqe_err
0,"Baseline (JW, 12q)",12,92,1572,3180,631,136,0.0,None


## 3. Reducing qubits, one trick at a time
For each step we record the resources and the **exact energy of the reduced problem**, so we can see how much accuracy each trick costs. We only run VQE on the small final candidates.

| Step | Trick | Why it's allowed |
|---|---|---|
| A | **Freeze core** | Li's 1s electrons sit far below everything else (−2.35 Ha) and barely take part in bonding |
| B | **Parity mapping** | conserving electron number lets two qubits be removed for free |
| C | **Symmetry tapering** | the molecule's mirror symmetries fix more qubit values, so those can go too |
| D | **Active space** | drop orbitals that can't take part in the bond |

In [4]:
def add(label, problem, mapper):
    op = mapper.map(problem.hamiltonian.second_q_op())
    row = measure_resources(build_ansatz(problem, mapper), op, label)
    row["reduced_exact_err"] = abs(exact_energy(problem, mapper) - E_EXACT)
    row["vqe_err"] = None
    results.append(row)
    return row

fc = FreezeCoreTransformer().transform(full)
add("A: freeze core (JW)", fc, JordanWignerMapper())
add("A+B: + parity", fc, ParityMapper(num_particles=fc.num_particles))
p6, m6 = reduced_lih("6q")
add("A+B+C: + tapering (6q)", p6, m6)
pd.DataFrame(results)

,config,qubits,params,CNOTs,depth,pauli_terms,meas_groups,reduced_exact_err,vqe_err
0,"Baseline (JW, 12q)",12,92,1572,3180,631,136,0.000000,None
1,A: freeze core (JW),10,24,346,698,276,58,0.000228,None
2,A+B: + parity,8,24,410,701,276,76,0.000228,None
3,A+B+C: + tapering (6q),6,10,118,242,231,72,0.000228,None


### Step D: choosing the active space (pick carefully!)
After freezing the core, the 5 remaining orbitals are: σ (bonding), σ\*, **π_x, π_y** (a degenerate pair), and a high σ orbital.

The obvious move is to keep the *lowest* orbitals. That keeps the π pair and throws away the high σ orbital, and it **fails**: the error is about 19 mHa, over ten times chemical accuracy.

The π orbitals point sideways, perpendicular to the Li–H bond, so by symmetry they can't mix with the σ bond. **Dropping the π pair instead** keeps the error under 1.6 mHa. This is a good point to make in the presentation.

In [5]:
naive = ActiveSpaceTransformer(num_electrons=2, num_spatial_orbitals=3).transform(fc)
add("D (naive): lowest 3 orbitals (3q)", naive,
    naive.get_tapered_mapper(ParityMapper(num_particles=naive.num_particles)))
p4, m4 = reduced_lih("4q")
add("A+B+C+D: drop π pair (4q)", p4, m4)
pd.DataFrame(results)

,config,qubits,params,CNOTs,depth,pauli_terms,meas_groups,reduced_exact_err,vqe_err
0,"Baseline (JW, 12q)",12,92,1572,3180,631,136,0.000000,None
1,A: freeze core (JW),10,24,346,698,276,58,0.000228,None
2,A+B: + parity,8,24,410,701,276,76,0.000228,None
3,A+B+C: + tapering (6q),6,10,118,242,231,72,0.000228,None
4,D (naive): lowest 3 orbitals (3q),3,4,18,53,34,13,0.019405,None
5,A+B+C+D: drop π pair (4q),4,8,72,162,100,25,0.001252,None


## 4. VQE on the reduced candidates
We run VQE on the 6- and 4-qubit versions to confirm UCCSD actually reaches the reduced problem's ground state.

In [6]:
for label, (p, m) in {"A+B+C: + tapering (6q)": (p6, m6),
                      "A+B+C+D: drop π pair (4q)": (p4, m4)}.items():
    e = run_vqe(p, m, build_ansatz(p, m))
    for r in results:
        if r["config"] == label:
            r["vqe_err"] = abs(e - E_EXACT)
    print(f"{label:28s} VQE = {e:.6f} Ha   error = {abs(e - E_EXACT)*1e3:.3f} mHa")

A+B+C: + tapering (6q)       VQE = -7.882096 Ha   error = 0.228 mHa


A+B+C+D: drop π pair (4q)    VQE = -7.881072 Ha   error = 1.252 mHa


## 5. Results table (for Person 4)

In [7]:
df = pd.DataFrame(results)
df["within_chem_acc"] = df["reduced_exact_err"] < CHEM_ACC
df["reduced_exact_err_mHa"] = (df["reduced_exact_err"] * 1e3).round(3)
df["vqe_err_mHa"] = (df["vqe_err"].astype(float) * 1e3).round(3)
df = df.drop(columns=["reduced_exact_err", "vqe_err"])
df.to_csv("p1_results.csv", index=False)
df

,config,qubits,params,CNOTs,depth,pauli_terms,meas_groups,within_chem_acc,reduced_exact_err_mHa,vqe_err_mHa
0,"Baseline (JW, 12q)",12,92,1572,3180,631,136,True,0.000,NaN
1,A: freeze core (JW),10,24,346,698,276,58,True,0.228,NaN
2,A+B: + parity,8,24,410,701,276,76,True,0.228,NaN
3,A+B+C: + tapering (6q),6,10,118,242,231,72,True,0.228,0.228
4,D (naive): lowest 3 orbitals (3q),3,4,18,53,34,13,False,19.405,NaN
5,A+B+C+D: drop π pair (4q),4,8,72,162,100,25,True,1.252,1.252


## 6. Summary and handover
- **12 → 6 qubits** (freeze core + parity + tapering): error about 0.23 mHa, comfortably accurate. CNOTs drop from 1572 to 118.
- **12 → 4 qubits** (also dropping the π pair): error about 1.25 mHa, just inside chemical accuracy. Only 72 CNOTs.
- Picking the active space naively breaks accuracy, so *which* orbitals you drop matters more than how many.

**For teammates:**
- Person 2 (ansatz): start from `reduced_lih("6q")`, which has room to spare on accuracy, then try `"4q"`.
- Person 3 (measurement): the 4-qubit operator has 100 Pauli terms in 25 qubit-wise groups, compared with 136 groups for the baseline.
- Person 4: the numbers are in `p1_results.csv`.

**Possible extras if there's time:** repeat at other bond lengths with `reduced_lih(level, bond_length=...)` to check the 4-qubit version stays accurate along the curve. Also compare Bravyi-Kitaev with parity.